In [1]:
print('he')

he


In [3]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [19]:
q1 = 'How can I run Docker?'
q2 = ''

In [20]:
v1 = model.encode(q1)

In [9]:
from ingest import load_faq_data

documents = load_faq_data()

In [10]:
texts = [doc['question'] + ' ' + doc['answer'] for doc in documents]

In [12]:
import os
from dotenv import load_dotenv

In [13]:
import numpy as np
from tqdm import tqdm

batch_size = 50
vectors = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i + batch_size]
    batch_vectors = model.encode(batch)
    vectors.extend(batch_vectors)

vectors = np.array(vectors)

100%|██████████| 25/25 [00:59<00:00,  2.37s/it]


In [14]:
len(vectors)

1208

In [16]:
vectors[10].shape

(384,)

In [21]:
import numpy as np

In [ ]:
scores = []

for i in range(len(vectors)):
    score = v1.dot(vectors[i])
    scores.append(score)

In [22]:
X = np.array(vectors)

In [23]:
scores = X.dot(v1)

In [24]:
idx = np.argmax(scores)
idx,scores[idx]

(np.int64(238), np.float32(0.6457993))

In [25]:
documents[238]

{'id': '6d91bd8f56',
 'course': 'machine-learning-zoomcamp',
 'section': 'Module 5. Deploying Machine Learning Models',
 'question': 'Docker: Trying to run a docker image I built but it says it’s unable to start the container process',
 'answer': 'Ensure that you used `pipenv` to install the necessary modules including `gunicorn`. Follow these steps:\n\n1. Use `pipenv shell` to enter the virtual environment.\n2. Build and run your Docker image.\n\nMake sure all dependencies are correctly specified in your Pipfile.'}

In [32]:
top5 = np.argsort(-scores)[:5]
# top5 = top5[::-1]

In [33]:
for idx in top5:
    print(scores[idx])
    print(documents[idx])
    print()

0.6457993
{'id': '6d91bd8f56', 'course': 'machine-learning-zoomcamp', 'section': 'Module 5. Deploying Machine Learning Models', 'question': 'Docker: Trying to run a docker image I built but it says it’s unable to start the container process', 'answer': 'Ensure that you used `pipenv` to install the necessary modules including `gunicorn`. Follow these steps:\n\n1. Use `pipenv shell` to enter the virtual environment.\n2. Build and run your Docker image.\n\nMake sure all dependencies are correctly specified in your Pipfile.'}

0.6455369
{'id': 'b2eabcd7dc', 'course': 'data-engineering-zoomcamp', 'section': 'Module 1: Docker', 'question': 'Docker: Setting up Docker on Mac', 'answer': 'For setting up Docker on macOS, you have two main options:\n\n1. **Download from Docker Website:**\n   - Visit the official Docker website and download the Docker Desktop for Mac as a `.dmg` file. This method is generally reliable and avoids issues related to licensing changes.\n\n2. **Using Homebrew:**\n   - 

In [35]:
from minsearch import VectorSearch

vindex = VectorSearch(keyword_fields=['course'])
vindex.fit(X, documents)

In [36]:
vindex.search(v1,num_results=5)

[{'id': '6d91bd8f56',
  'course': 'machine-learning-zoomcamp',
  'section': 'Module 5. Deploying Machine Learning Models',
  'question': 'Docker: Trying to run a docker image I built but it says it’s unable to start the container process',
  'answer': 'Ensure that you used `pipenv` to install the necessary modules including `gunicorn`. Follow these steps:\n\n1. Use `pipenv shell` to enter the virtual environment.\n2. Build and run your Docker image.\n\nMake sure all dependencies are correctly specified in your Pipfile.'},
 {'id': 'b2eabcd7dc',
  'course': 'data-engineering-zoomcamp',
  'section': 'Module 1: Docker',
  'question': 'Docker: Setting up Docker on Mac',
  'answer': 'For setting up Docker on macOS, you have two main options:\n\n1. **Download from Docker Website:**\n   - Visit the official Docker website and download the Docker Desktop for Mac as a `.dmg` file. This method is generally reliable and avoids issues related to licensing changes.\n\n2. **Using Homebrew:**\n   - Be

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import os

# Load .env file
load_dotenv()

# Read API key
api_key = os.getenv("OPENAI_API_KEY")

# Initialize client
client = OpenAI(api_key=api_key)

# Example request
response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=[
        {"role": "user", "content": "Hello!"}
    ]
)

print(response.choices[0].message.content)

Hello! How can I assist you today?


In [38]:
from ingest import build_index
index = build_index(documents)

In [54]:
from rag_helper import RAGBase

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    instructions=instructions,
    index=index,    
    llm_client=client,
    course = "mlops-zoomcamp"
)

In [47]:
assistant.rag(q1)

'The context doesn’t include a general “how to run Docker” command. It only mentions running Docker-related tasks like rebuilding and starting a Docker Compose app with:\n\n```bash\ndocker-compose up --build\n```\n\nIf you meant something else by “run Docker,” please share the specific Docker task you want to do.'

In [ ]:
class RAGVector(RAGBase):

    def __init__(self, embedder, **kwargs):
        super().__init__(**kwargs)
        self.embedder = embedder

    def search(self,query,num_results=5):
        query_vector = self.embedder.encode(query)
        filter_dict = {'course':self.course}

        return self.index.search(
            query_vector,
            num_results=num_results,
            filter_dict=filter_dict
        )

In [62]:
course='mlops-zoomcamp'

In [64]:
vector_assistant = RAGVector(
    instructions=instructions,
    embedder=model,
    index=vindex,
    llm_client=client,
    course=course
)

In [52]:
vector_assistant.rag(q1)

'To run Docker in the context of this FAQ, use:\n\n```bash\ndocker-compose up --build\n```\n\nIf you’re trying to run a Streamlit app in Docker and get an error saying `streamlit` is not found, make sure you have:\n\n1. Created a `Dockerfile`\n2. Added `streamlit` to the `docker-compose` configuration\n3. Rebuilt and started the app with the command above'

In [65]:
q2 = "How do I run Docker on Windows?"


In [58]:
print(assistant.rag(q2))

If you’re on Windows and want to run Docker in WSL2 **without Docker Desktop**, follow these steps:

1. Install Docker:
```bash
curl -fsSL https://get.docker.com -o get-docker.sh
sudo sh get-docker.sh
```

2. Add your user to the Docker group:
```bash
sudo usermod -aG docker $USER
```

3. Enable the Docker service:
```bash
sudo systemctl enable docker.service
```

4. Test the installation:
```bash
docker --version
docker compose version
docker run hello-world
```

5. If Docker does not start automatically after restarting WSL, add this to your `.profile` or `.zprofile`:
```bash
if grep -q "microsoft" /proc/version > /dev/null 2>&1; then
   if service docker status 2>&1 | grep -q "is not running"; then
      wsl.exe --distribution "${WSL_DISTRO_NAME}" --user root \
      --exec /usr/sbin/service docker start > /dev/null 2>&1
   fi
fi
```

If you meant **Docker Compose**, note that **Compose v1 is deprecated from April 2023 onwards**.


In [66]:
print(vector_assistant.rag(q2))

To run Docker on Windows, use WSL2. You can install Docker in WSL2 without Docker Desktop by doing this:

1. Install Docker:
```bash
curl -fsSL https://get.docker.com -o get-docker.sh
sudo sh get-docker.sh
```

2. Add your user to the Docker group:
```bash
sudo usermod -aG docker $USER
```

3. Enable the Docker service:
```bash
sudo systemctl enable docker.service
```

4. Test it:
```bash
docker --version
docker compose version
docker run hello-world
```

5. If Docker does not start automatically after restarting WSL, add this to your `.profile` or `.zprofile`:
```bash
if grep -q "microsoft" /proc/version > /dev/null 2>&1; then
   if service docker status 2>&1 | grep -q "is not running"; then
      wsl.exe --distribution "${WSL_DISTRO_NAME}" --user root \
      --exec /usr/sbin/service docker start > /dev/null 2>&1
   fi
fi
```

If you want to use WSL on Windows, you can also install it with:
```bash
wsl --install
```
